In [1]:
import pandas as pd
import re

In [2]:
# Define columns needed and create dataframes from IPEDS 2011 and 2021
cols = ["UNITID", "CIPCODE", "AWLEVEL", "CTOTALT", "CTOTALM", "CTOTALW"]
c2011_clean = pd.read_csv(
    r"C:\capstone_data\raw\c2011_a.csv", 
    usecols=cols,
    dtype={"CIPCODE":str}
)

In [3]:
c2021_clean = pd.read_csv(
    r"C:\capstone_data\raw\c2021_a.csv", 
    usecols=cols,
    dtype={"CIPCODE":str}
)

In [4]:
# Dictionary for renaming columns
rename_dict = {
    "UNITID" : "unitid"
    ,"CIPCODE" : "cipcode"
    ,"AWLEVEL" : "credlev"
    ,"CTOTALT" : "completions"
    ,"CTOTALM" : "completions_men"
    ,"CTOTALW" : "completions_women"
}

In [5]:
# Rename the columns
c2011_clean = c2011_clean.rename(columns=rename_dict)
c2021_clean = c2021_clean.rename(columns=rename_dict)

In [6]:
print(c2011_clean.columns == c2021_clean.columns)

[ True  True  True  True  True  True]


In [7]:
# Set years
c2011_clean["year"] = 2011
c2021_clean["year"] = 2021

In [8]:
# "Stack" the two sets of data so that 2011 sits on top of 2021
completions = pd.concat([c2011_clean, c2021_clean], ignore_index=True)
completions.shape

(562151, 7)

In [9]:
c2011_clean["credlev"].value_counts().sort_index()
c2021_clean["credlev"].value_counts().sort_index()

credlev
2      30085
3      48506
4       2578
5     103820
6      12450
7      43913
8       5179
17     13130
18      2866
19       383
20      3762
21     29671
Name: count, dtype: int64

In [10]:
# Create a dictionary for the credlev codes, limiting it to the ones we are interested in
level_map = {
    2 : "Short Certificate"
    ,3 : "Long Certificate"
    ,5 : "Long Certificate"
    ,4 : "Associate"
    ,6 : "Bachelor"
}
completions["credential_group"] = completions["credlev"].map(level_map)

In [11]:
# Format ipeds cip code to xxxx dtype str format. Maintain cip code 99 for accuracy.
def ipeds_to_4digit(cip):
    s = str(cip)
    digits = "".join(re.findall(r"\d+", s))    # Extract the digits
    if digits == "99":
        return "9900"                         # Special: cip 99 to 9900
    digits = digits.zfill(6)
    return digits [:4]

completions["cip4"] = completions["cipcode"].apply(ipeds_to_4digit)

In [12]:
# Checkpoint to see if everything is looking good.
completions["year"].value_counts()

year
2021    296343
2011    265808
Name: count, dtype: int64

In [13]:
completions["credlev"].value_counts()

credlev
5     207121
3      96997
7      79758
2      59204
21     29671
1      25020
17     23747
6      17565
8       8312
4       5521
18      4794
20      3762
19       679
Name: count, dtype: int64

In [14]:
completions.head(8)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year,credential_group,cip4
0,100636,09.0999,3,66,34,32,2011,Long Certificate,0909
1,100636,10.0105,3,547,398,149,2011,Long Certificate,1001
2,100636,11.0101,3,69,65,4,2011,Long Certificate,1101
3,100636,11.0401,3,915,705,210,2011,Long Certificate,1104
4,100636,13.0499,3,269,144,125,2011,Long Certificate,1304
5,100636,13.9999,3,1486,1244,242,2011,Long Certificate,1399
6,100636,13.9999,4,993,812,181,2011,Associate,1399
7,100636,15.0303,3,1012,913,99,2011,Long Certificate,1503


In [15]:
completions.tail(8)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year,credential_group,cip4
562143,497329,51.0801,2,0,0,0,2021,Short Certificate,5108
562144,497329,51.3501,21,0,0,0,2021,NaN,5135
562145,497329,99,2,0,0,0,2021,Short Certificate,9900
562146,497329,99,21,0,0,0,2021,NaN,9900
562147,497338,51.0801,21,0,0,0,2021,NaN,5108
562148,497338,51.3801,3,0,0,0,2021,Long Certificate,5138
562149,497338,99,3,0,0,0,2021,Long Certificate,9900
562150,497338,99,21,0,0,0,2021,NaN,9900


In [16]:
completions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562151 entries, 0 to 562150
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   unitid             562151 non-null  int64 
 1   cipcode            562151 non-null  object
 2   credlev            562151 non-null  int64 
 3   completions        562151 non-null  int64 
 4   completions_men    562151 non-null  int64 
 5   completions_women  562151 non-null  int64 
 6   year               562151 non-null  int64 
 7   credential_group   386408 non-null  object
 8   cip4               562151 non-null  object
dtypes: int64(6), object(3)
memory usage: 38.6+ MB


In [17]:
completions.dtypes

unitid                int64
cipcode              object
credlev               int64
completions           int64
completions_men       int64
completions_women     int64
year                  int64
credential_group     object
cip4                 object
dtype: object

In [18]:
completions["cip4"].unique()

array(['0909', '1001', '1101', '1104', '1304', '1399', '1503', '1504',
       '1505', '1506', '1507', '1510', '2203', '2609', '2902', '2904',
       '2999', '3106', '3199', '4004', '4199', '4301', '4302', '4303',
       '4407', '4706', '4901', '4999', '5006', '5009', '5106', '5107',
       '5108', '5109', '5110', '5115', '5118', '5122', '5131', '5199',
       '5202', '5203', '5210', '5401', '9900', '0100', '0101', '0109',
       '0110', '0111', '0199', '0305', '0403', '1002', '1310', '1312',
       '1313', '1408', '1410', '1419', '1499', '1502', '1508', '1901',
       '2301', '2401', '2601', '2701', '4005', '4008', '4201', '4228',
       '4506', '4510', '4511', '5007', '5102', '5208', '5214', '0502',
       '0901', '1301', '1311', '1401', '1405', '1409', '1414', '1418',
       '1601', '2602', '2604', '2605', '2608', '2610', '2611', '2613',
       '2615', '2703', '3019', '3099', '3105', '3801', '4404', '4502',
       '4599', '5001', '5005', '5104', '5105', '5112', '5117', '5123',
      